Inputs to the analyis: `image.tif`, `pca.tif`, `msu_pts.gpkg`
* Read in all inputs.
* Identify smaller AOI tile.
* Extract DINOv3 embedding for the window that contains the MSU tree point.
* Extract PCA vector from pca.tif at the MSU tree point location.
* Create table with x, y, species, token_row, token_col, pca_vec, embed_vec
* Create two pairs of sets: within species, across species
* Compute cosine distance for some pairs
* Create visualization to show results

test site: `kev_2023-07-21`

In [ ]:
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path("../dinov3").resolve()))
sys.path.insert(0, str(Path('../src').resolve()))

import cosine_similarity as cs

%load_ext autoreload
%autoreload 2

## Identify smaller AOI tile

In [ ]:
cs.aoi_clip(
    image_path="../data/msu_images/kev_2023-07-21_cropped.tif",
    pca_path="../data/pca/kev_2023-07-21_cropped_pca.tif",
    trees_path="../data/msu_field/_clean/kev_fielddata_clean.gpkg",
    img_out_path="../data/tmp/kev_2023-07-21_maxar_aoi.tif",
    pca_out_path="../data/tmp/kev_2023-07-21_pca_aoi.tif",
    buffer_m=30.0,
)

## Extract features for each tree location

In [ ]:
results = cs.per_tree_features("../data/tmp/kev_2023-07-21_maxar_aoi.tif",
                              "../data/tmp/kev_2023-07-21_pca_aoi.tif",
                              "../data/msu_field/_clean/kev_fielddata_clean.gpkg",)

## Calculate similarity


### Interpretation
* distance ≈ 0.0 → almost identical vectors, same species pairs should have lower distances
* distance ≈ 1.0 → unrelated vectors, different species pairs should have higher distances

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Take the first few trees for a simple test
n = -10
subset = results 

# Extract species and embeddings
species = [rec["species"] for rec in subset]
embeddings = np.stack([rec["embed_vec"] for rec in subset])   

print("Species in test subset:", species)
print("Embeddings shape:", embeddings.shape) # shape (n_species, 768)

sim_matrix = cosine_similarity(embeddings)   # (5 × 5) matrix
dist_matrix = 1 - sim_matrix

print("\nCosine similarity matrix:")
print(sim_matrix)
print("\nCosine distance matrix:")
print(dist_matrix)

print("\nPairwise cosine distances:")
for i in range(len(subset)):
    for j in range(i+1, len(subset)):
        print(f"Tree {i} ({species[i]}) ↔ Tree {j} ({species[j]}): "
              f"{dist_matrix[i, j]:.4f}")


In [ ]:
def plot_cosine_distance_heatmap(dist_matrix, species, title="Cosine distance heatmap"):
    """
    Plot a heatmap of cosine distances between trees.

    Parameters
    ----------
    dist_matrix : np.ndarray
        Square matrix (N x N) of cosine distances (0 = identical, ~1 = orthogonal).
    species : list or array-like of str
        Species labels for each tree, length N.
    title : str, optional
        Title for the plot.
    """
    dist_matrix = np.asarray(dist_matrix)
    n = dist_matrix.shape[0]

    if dist_matrix.shape[0] != dist_matrix.shape[1]:
        raise ValueError("dist_matrix must be square (N x N).")
    if len(species) != n:
        raise ValueError("len(species) must match dist_matrix size.")

    # Build concise labels like "0: SpeciesA", "1: SpeciesB"
    labels = [f"{i}: {sp}" for i, sp in enumerate(species)]

    plt.figure(figsize=(16, 15))
    im = plt.imshow(dist_matrix, interpolation="nearest")  # default colormap

    plt.title(title)
    plt.xlabel("Tree index / species")
    plt.ylabel("Tree index / species")

    plt.xticks(ticks=np.arange(n), labels=labels, rotation=90)
    plt.yticks(ticks=np.arange(n), labels=labels)

    plt.colorbar(im, label="Cosine distance (1 - similarity)")
    plt.tight_layout()
    plt.show()


In [ ]:
#plot_cosine_distance_heatmap(dist_matrix, species)